# <center>Data Preparation: iOS Apps<center>

In [56]:
import pandas as pd
import ast
from collections import Counter
import numpy as np



### 1. Load dataset


In [91]:
ios=pd.read_csv('appsIos.csv')
ios_dummy=pd.read_csv('appsIos.csv')
print(f'Loaded {len(ios_dummy)} iOS apps across {ios_dummy['genre'].nunique()} genres.')

pd.set_option('display.max_columns',None)
print(ios_dummy.head(5))

Loaded 11964 iOS apps across 23 genres.
                             app      id                 title  rating  \
0         br.com.msbtec.midiamix  544454              MidiaMix     NaN   
1              com.xingling01.cs  250299  星灵骑士团-FATE蒸汽与魔法的大千世界     NaN   
2            com.88DESIGNS.Mzali  250497                 Mzali     NaN   
3  com.nicovecciaterlier.working  250455    Nicoveccia Atelier     NaN   
4               com.mshiphoneold  544794           MSH SERVICE     NaN   

   num_reviews      genre family_genre     updated    released offersiap  \
0          NaN       News          NaN  2019-10-13  2019-10-12     False   
1          NaN      Games          NaN  2018-04-22  2018-04-22      True   
2          NaN  Education          NaN  2019-01-14  2017-01-24     False   
3          NaN  Lifestyle          NaN  2019-12-12  2019-12-11     False   
4          NaN   Business          NaN  2019-08-31  2017-05-04     False   

                                          frameworks  \
0 


### 2. Drop unnecessary columns

We will drop the redundant columns 'CalendarRead', 'CalendarWrite', 'ContactsRead', and 'ContactsWrite' as whenever the ‘Contacts’/‘Calender’ is True, both Read and Write respectively are True. The column ‘family_genre’ is almost entirely null, so we will drop it too. The ‘permissions’ column will also be dropped, as we already have binary columns for permissions.

In [92]:
drop_cols=['CalendarRead','CalendarWrite','ContactsRead','ContactsWrite','family_genre', 'permissions']

ios=ios.drop(columns=drop_cols,errors='ignore')
print(f'Dropped {len(drop_cols)} columns:\n {drop_cols}')

Dropped 6 columns:
 ['CalendarRead', 'CalendarWrite', 'ContactsRead', 'ContactsWrite', 'family_genre', 'permissions']



### 3. Parsing the columns containing string-encoded lists and engineering SDK-level features from the 'frameworks' column


Moving on to columns like ‘frameworks’ and ‘trackersettings’, we see that although the entries look like lists, they are actually just simple text strings. We will convert them to Python lists using ast.literal_eval so we can actually perform operations on them. However, since ast.literal_eval alone would crash on empty or malformed entries, we will implement a function using ast.literal_eval which returns an empty list on failure.

In [93]:
def parse_list(entry):
    if pd.isna(entry) or list in ('','[]'):
        return []
    try:
        result=ast.literal_eval(entry)
        return result if isinstance(result,list) else []
    except (ValueError,SyntaxError):
        return []

ios['frameworks_list']=ios['frameworks'].apply(parse_list)
ios['trackersettings_list']=ios['trackersettings'].apply(parse_list)

print(f"\nParsed list columns:")
print(f"'frameworks': {ios['frameworks_list'].apply(len).gt(0).sum():.1f} apps with entries")
print(f"'trackersettings': {ios['trackersettings_list'].apply(len).gt(0).sum()} apps with entries")


Parsed list columns:
'frameworks': 11677.0 apps with entries
'trackersettings': 249 apps with entries


Now we will extract useful signals from the transformed frameworks column. Each entry in a frameworks list is an iOS system framework, which is a named Apple library the app has linked against. Looking at this tells us what the app actually has access to at an OS or hardware level, independent of what it claimed in its permission strings. However, we will not consider all frameworks. Some frameworks like Foundation, CoreGraphics, etc. are present in almost all apps and have no discriminatory power, whereas some like SwiftUI, PencilKit, etc. are too rarely present to matter in the broad analysis. We will not consider them, along with the ones related to graphics, game engine, etc., which are not relevant to privacy. For the remaining ones, we will create a binary column for each: True if that framework appears in the app's list, False if not. Additionally, we will also handle the tracker opt-out flags from ‘trackersettings’. Only a relatively small percentage of apps have entries like 'Firebase_Messaging_Delayed' and 'Firebase_Analytics_Deactivated' which shows that these apps are deliberately limiting tracking, likely to comply with the guidelines of policies like GDPR (General Data Protection Regulation).



In [95]:
frameworks_cnt=Counter()
for fw_list in ios['frameworks_list']:
    frameworks_cnt.update(fw_list)

print(f"\n{len(frameworks_cnt)} unique frameworks in dataset (sorted by frequency):")
for fw,count in sorted(frameworks_cnt.items(),key=lambda x:-x[1]):
    print(f"{fw:>30}  Count: {count}, Percentage: ({100*count/len(ios):.1f}%)")


117 unique frameworks in dataset (sorted by frequency):
                    Foundation  Count: 11677, Percentage: (97.6%)
                         UIKit  Count: 11666, Percentage: (97.5%)
                CoreFoundation  Count: 11565, Percentage: (96.7%)
                  CoreGraphics  Count: 11356, Percentage: (94.9%)
           SystemConfiguration  Count: 10482, Percentage: (87.6%)
                    QuartzCore  Count: 10080, Percentage: (84.3%)
                      Security  Count: 9979, Percentage: (83.4%)
                  AVFoundation  Count: 9281, Percentage: (77.6%)
            MobileCoreServices  Count: 8286, Percentage: (69.3%)
                  AudioToolbox  Count: 7963, Percentage: (66.6%)
                  CoreLocation  Count: 7756, Percentage: (64.8%)
                      StoreKit  Count: 7740, Percentage: (64.7%)
                     CFNetwork  Count: 7582, Percentage: (63.4%)
                 CoreTelephony  Count: 7496, Percentage: (62.7%)
                     CoreMe

In [96]:
#frameworks retained
frameworks_retained={
    #All frameworks are present in Apple Developer Documentation except Twitter (third-party)and iAd (documentation no longer available)

    #Advertising and tracking related
    'AdSupport':'fw_ad_support','iAd':'fw_iad','SafariServices':'fw_safari_services','DeviceCheck':'fw_device_check',

    #Access to user data
    'Contacts':'fw_contacts','ContactsUI':'fw_contacts_ui',
    'MessageUI':'fw_message_ui','Messages':'fw_messages',
    'CallKit':'fw_call_kit',
    'AddressBook':'fw_address_book','AddressBookUI':'fw_address_book_ui',
    'EventKit':'fw_event_kit','EventKitUI':'fw_event_kit_ui',
    'Accounts':'fw_accounts',
    'Photos':'fw_photos','PhotosUI':'fw_photos_ui','AssetsLibrary':'fw_assets_library',#Legacy photo library API (pre-iOS 8)
    'Social':'fw_social',
    'Intents':'fw_intents',

    #Location-related
    'CoreLocation':'fw_core_location','MapKit':'fw_map_kit',

    #Audio, camera and media related
    'Speech':'fw_speech','AVFoundation':'fw_av_foundation','MediaPlayer':'fw_media_player',

    #Hardware and sensor related
    'CoreBluetooth':'fw_core_bluetooth','CoreMotion':'fw_core_motion','CoreNFC':'fw_nfc',

    #Web and network related
    'WebKit':'fw_webkit','NetworkExtension':'fw_network_extension',
    'MultipeerConnectivity':'fw_multipeer',
    'WatchConnectivity':'fw_watch_connectivity',

    'HealthKit':'fw_health_kit', #health
    'LocalAuthentication':'fw_local_auth', #biometrics

    'StoreKit':'fw_store_kit', #in-app purchases
    'PassKit':'fw_pass_kit', #apple payments

    #Notifications and analytics related
    'UserNotifications':'fw_notifications','PushKit':'fw_push_kit','CoreTelephony':'fw_core_telephony',

    #Cloud, authorization
    'CloudKit':'fw_cloud_kit','AuthenticationServices':'fw_auth_services',

    'Twitter':'fw_twitter',
    'HomeKit':'fw_home_kit', #smart home access
    'CoreML':'fw_core_ml', #Ml models on device
    'Vision':'fw_vision', #image/face analysis
    'ARKit':'fw_arkit', #augmented reality
}

f=[]
for framework,col in frameworks_retained.items():
    ios[col]=ios['frameworks_list'].apply(lambda x:framework in x)
    f.append(framework)

print(f"\n{len(f)} framework-based feature columns:\n{f}")


45 framework-based feature columns:
['AdSupport', 'iAd', 'SafariServices', 'DeviceCheck', 'Contacts', 'ContactsUI', 'MessageUI', 'Messages', 'CallKit', 'AddressBook', 'AddressBookUI', 'EventKit', 'EventKitUI', 'Accounts', 'Photos', 'PhotosUI', 'AssetsLibrary', 'Social', 'Intents', 'CoreLocation', 'MapKit', 'Speech', 'AVFoundation', 'MediaPlayer', 'CoreBluetooth', 'CoreMotion', 'CoreNFC', 'WebKit', 'NetworkExtension', 'MultipeerConnectivity', 'WatchConnectivity', 'HealthKit', 'LocalAuthentication', 'StoreKit', 'PassKit', 'UserNotifications', 'PushKit', 'CoreTelephony', 'CloudKit', 'AuthenticationServices', 'Twitter', 'HomeKit', 'CoreML', 'Vision', 'ARKit']


In [97]:
#trackersettings flags handling
print("Unique tracker flags present:")
print(ios['trackersettings_list'].explode().dropna().unique())

tracker_flags={
    'has_firebase_messaging_delayed':lambda x:any('Firebase_Messaging_Delayed' in i for i in x),
    'has_fb_events_delayed':lambda x:any('FB_Events_Delayed' in i for i in x),
    'has_fb_adid_disabled':lambda x:any('FB_ADID_Disabled' in i for i in x),
    'has_firebase_analytics_deactivated':lambda x:any('Firebase_Analytics_Deactivated' in i for i in x),
    'has_gads_init_delayed':lambda x:any('GAds_Init_Delayed' in i for i in x),
    'has_fb_init_delayed':lambda x:any('FB_Init_Delayed' in i for i in x),
    'has_firebase_analytics_disabled':lambda x:any('Firebase_Analytics_Disabled' in i for i in x),
    'has_firebase_idfv_disabled':lambda x:any('Firebase_IDFV_Disabled' in i for i in x),
    'has_firebase_analytics_ads_disabled':lambda x:any('Firebase_Analytics_Ads_Disabled' in i for i in x)
}

#create columns
for col,func in tracker_flags.items():
    ios[col]=ios['trackersettings_list'].apply(func)

print('\nNumber of apps having each tracker flag:')
for col in tracker_flags.keys():
    print(f"{col}: {ios[col].sum()} apps")

#dropping columns we don't need anymore as we have extracted the features
ios.drop(columns=['frameworks','trackersettings','frameworks_list','trackersettings_list'],inplace=True)
print("\nDataset now:")
print(ios.head(5))

Unique tracker flags present:
['Firebase_Messaging_Delayed' 'FB_Events_Delayed' 'FB_ADID_Disabled'
 'Firebase_Analytics_Deactivated' 'GAds_Init_Delayed' 'FB_Init_Delayed'
 'Firebase_Analytics_Disabled' 'Firebase_IDFV_Disabled'
 'Firebase_Analytics_Ads_Disabled']

Number of apps having each tracker flag:
has_firebase_messaging_delayed: 25 apps
has_fb_events_delayed: 147 apps
has_fb_adid_disabled: 98 apps
has_firebase_analytics_deactivated: 19 apps
has_gads_init_delayed: 62 apps
has_fb_init_delayed: 31 apps
has_firebase_analytics_disabled: 24 apps
has_firebase_idfv_disabled: 3 apps
has_firebase_analytics_ads_disabled: 2 apps

Dataset now:
                             app      id                 title  rating  \
0         br.com.msbtec.midiamix  544454              MidiaMix     NaN   
1              com.xingling01.cs  250299  星灵骑士团-FATE蒸汽与魔法的大千世界     NaN   
2            com.88DESIGNS.Mzali  250497                 Mzali     NaN   
3  com.nicovecciaterlier.working  250455    Nicoveccia Atel


### 4. Handle missing and duplicate data

Handling missing data:
- The columns ‘rating’ and ‘num_reviews’ are too sparse for general analysis, but we will keep the columns and add a ‘has_rating’ boolean column so we can easily filter the apps during any further analysis.
- The ‘offersiap’ column records whether an app offers in-app purchases. Some apps have null values, but we cannot say for sure that it is because they definitely have no IAP. So we will fill those cells with 'False' as a conservative assumption since it is not a big enough number (318 entries or 2.7%) to meaningfully affect results.
- The date columns ‘released’ and ‘updated’ are stored as strings, so we will convert them to proper datetime objects. Then we will compute the gap between release and last update as 'app_age_days'. This indicates how actively maintained an app is and can later be used to analyze whether data collection is different for newer or more actively updated apps.

We also check for duplicate records such that each app corresponds to only one record.

In [98]:
#'rating' and 'num_reviews'
ios['has_rating']=ios['rating'].notna()

# offersiap
ios['offersiap']=ios['offersiap'].fillna(False)
ios['offersiap']=ios['offersiap'].map({'True':True,'False':False,True:True,False:False})

# Parse dates
ios['released']=pd.to_datetime(ios['released'],errors='coerce')
ios['updated']=pd.to_datetime(ios['updated'],errors='coerce')
ios['app_age_days']=(ios['updated']-ios['released']).dt.days.clip(lower=0)

print(f'Number of duplicate records: {ios.duplicated(subset='id').sum()}')

Number of duplicate records: 0


/var/folders/g3/k4rvg0kd78n6zmp9fyt4swxr0000gn/T/ipykernel_58312/1347363671.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ios['offersiap']=ios['offersiap'].fillna(False)


### 5. Removing genres that do not meet a minimum threshold

The next step is to decide which genres to retain in the study. The novel per-app metric is based on the fact that each genre’s “normal” (or median) permission profile is meaningful. If a genre has very few apps, the median is not robust enough to be considered a reliable baseline, as a few unusual apps (i.e., outliers) would skew it dramatically. Thus, we will set a minimum threshold of 100 apps per genre. This gives reliable baselines while only dropping the two genres, ‘Weather’ and ‘Newsstand’ (i.e., 74 apps out of ~12000 apps, which is relatively nothing).

In [99]:
#number of apps in each genre
genre_counts=ios['genre'].value_counts()
print(f'Genre - Number of Apps: \n{genre_counts}')


Genre - Number of Apps: 
genre
Games                 1602
Business              1345
Education             1052
Utilities              996
Lifestyle              919
Health and Fitness     678
Entertainment          579
Travel                 531
Productivity           525
Finance                524
Food and Drink         411
Shopping               384
Sports                 366
Social                 322
Medical                314
Music                  303
News                   297
Photo and Video        252
Reference              184
Book                   160
Navigation             146
Weather                 59
Newsstand               15
Name: count, dtype: int64


In [100]:
threshold=100
valid_genres=genre_counts[genre_counts>=threshold].index.tolist()
dropped_genres=genre_counts[genre_counts<threshold].index.tolist()

ios_clean=ios[ios['genre'].isin(valid_genres)].copy()
ios_clean['genre']=ios_clean['genre'].astype('category')
print(f"Dropped {len(dropped_genres)} genres: {dropped_genres}")

Dropped 2 genres: ['Weather', 'Newsstand']



### 6. Defining column groups


We will be grouping certain columns so that in later analyses we can import and use them consistently. The 7 permission columns (Bluetooth, Calendar, Camera, Contacts, Location, Microphone and Motion), which act as the core inputs for the per-app metric and are shared with Android (to make future cross-platform extension easy), form a group. Another group consists of the 3 SDK columns (Google Firebase, AdID, Analytics), which are pre-encoded signals from the dataset authors. The framework features are also grouped together.

In [101]:
#permissions (shared with Android)
permission_cols=['Bluetooth','Calendar','Camera','Contacts','Location','Microphone','Motion']

#pre-encoded SDK signals
sdk_cols=['Google Firebase','AdID','Analytics']

#all features derived from 'frameworks'
framework_cols=list(frameworks_retained.values())

print(f"\nColumn groups defined:\n")
print(f"Permission columns:{permission_cols}\n")
print(f"SDK columns:{sdk_cols}\n")
print(f"Framework features: {len(framework_cols)} columns\n{framework_cols}")


Column groups defined:

Permission columns:['Bluetooth', 'Calendar', 'Camera', 'Contacts', 'Location', 'Microphone', 'Motion']

SDK columns:['Google Firebase', 'AdID', 'Analytics']

Framework features: 45 columns
['fw_ad_support', 'fw_iad', 'fw_safari_services', 'fw_device_check', 'fw_contacts', 'fw_contacts_ui', 'fw_message_ui', 'fw_messages', 'fw_call_kit', 'fw_address_book', 'fw_address_book_ui', 'fw_event_kit', 'fw_event_kit_ui', 'fw_accounts', 'fw_photos', 'fw_photos_ui', 'fw_assets_library', 'fw_social', 'fw_intents', 'fw_core_location', 'fw_map_kit', 'fw_speech', 'fw_av_foundation', 'fw_media_player', 'fw_core_bluetooth', 'fw_core_motion', 'fw_nfc', 'fw_webkit', 'fw_network_extension', 'fw_multipeer', 'fw_watch_connectivity', 'fw_health_kit', 'fw_local_auth', 'fw_store_kit', 'fw_pass_kit', 'fw_notifications', 'fw_push_kit', 'fw_core_telephony', 'fw_cloud_kit', 'fw_auth_services', 'fw_twitter', 'fw_home_kit', 'fw_core_ml', 'fw_vision', 'fw_arkit']



### 7. Save cleaned dataset


In [110]:
ios_clean=ios_clean[['app', 'id', 'title', 'has_rating','rating', 'num_reviews', 'genre', 'updated', 'released','app_age_days','offersiap', 'Google Firebase', 'AdID', 'Bluetooth', 'Calendar', 'Camera', 'Contacts', 'Location', 'Microphone', 'Motion', 'Analytics', 'fw_ad_support', 'fw_iad', 'fw_safari_services', 'fw_device_check', 'fw_contacts', 'fw_contacts_ui', 'fw_message_ui', 'fw_messages', 'fw_call_kit', 'fw_address_book', 'fw_address_book_ui', 'fw_event_kit', 'fw_event_kit_ui', 'fw_accounts', 'fw_photos', 'fw_photos_ui', 'fw_assets_library', 'fw_social', 'fw_intents', 'fw_core_location', 'fw_map_kit', 'fw_speech', 'fw_av_foundation', 'fw_media_player', 'fw_core_bluetooth', 'fw_core_motion', 'fw_nfc', 'fw_webkit', 'fw_network_extension', 'fw_multipeer', 'fw_watch_connectivity', 'fw_health_kit', 'fw_local_auth', 'fw_store_kit', 'fw_pass_kit', 'fw_notifications', 'fw_push_kit', 'fw_core_telephony', 'fw_cloud_kit', 'fw_auth_services', 'fw_twitter', 'fw_home_kit', 'fw_core_ml', 'fw_vision', 'fw_arkit', 'has_firebase_messaging_delayed', 'has_fb_events_delayed', 'has_fb_adid_disabled', 'has_firebase_analytics_deactivated', 'has_gads_init_delayed', 'has_fb_init_delayed', 'has_firebase_analytics_disabled', 'has_firebase_idfv_disabled', 'has_firebase_analytics_ads_disabled']]

ios_clean.to_csv('ios_clean.csv',index=False)

print(f"\n{'-' * 60}")
print(f"Initial Dataset: appsIos.csv")
print(f"Cleaned Dataset: ios_clean.csv")
print(f"{'-' * 60}")
print(f"Input: {len(ios):,} apps, 27 columns")
print(f"Output: {len(ios_clean):,} apps, {len(ios_clean.columns)} columns")
print(f"\nColumn summary:")
print(f"Core identifiers: app, id, title, genre, offersiap")
print(f"7 permission flags: {', '.join(permission_cols)}")
print(f"3 SDK signals:{', '.join(sdk_cols)}")
print(f"{len(framework_cols)} framework features:  fw_ad_support, fw_messages, etc.")
print(f"9 tracker opt-out: has_fb_events_delayed, has_firebase_analytics_deactivated, etc. ")
print(f"Temporal: released, updated, app_age_days")
print(f"Rating: has_rating, rating, num_reviews")


------------------------------------------------------------
Initial Dataset: appsIos.csv
Cleaned Dataset: ios_clean.csv
------------------------------------------------------------
Input: 11,964 apps, 27 columns
Output: 11,890 apps, 75 columns

Column summary:
Core identifiers: app, id, title, genre, offersiap
7 permission flags: Bluetooth, Calendar, Camera, Contacts, Location, Microphone, Motion
3 SDK signals:Google Firebase, AdID, Analytics
45 framework features:  fw_ad_support, fw_messages, etc.
9 tracker opt-out: has_fb_events_delayed, has_firebase_analytics_deactivated, etc. 
Temporal: released, updated, app_age_days
Rating: has_rating, rating, num_reviews


In [102]:
#Checking column datatypes
print("Column - Datatype:")
print(ios_clean.dtypes)

Column - Datatype:
app                                            object
id                                              int64
title                                          object
rating                                        float64
num_reviews                                   float64
genre                                        category
updated                                datetime64[ns]
released                               datetime64[ns]
offersiap                                        bool
Google Firebase                                  bool
AdID                                             bool
Bluetooth                                        bool
Calendar                                         bool
Camera                                           bool
Contacts                                         bool
Location                                         bool
Microphone                                       bool
Motion                                           bool
Analytics